# DePlag Fast Local Accuracy Notebook

This notebook is intentionally separate from `DePlag_Accuracy_Upgrade_Colab.ipynb`.

Design goal: faster training without throwing away accuracy. It uses local datasets from `data/raw`, excludes the recently added `paraphrased_articles.csv`, and starts from a paraphrase-trained cross-encoder instead of a raw base model.

## Brutal Engineering Choices

- Do not fine-tune a raw base model first. Use `cross-encoder/quora-roberta-base`, already trained for pair classification.
- Use local files: MSRP, PAWS, STS, SNLI.
- Exclude `paraphrased_articles.csv` because it was recently added and may distort the experiment.
- Balance all splits to avoid majority-class garbage accuracy.
- Train for one strong epoch first. Add more only if validation is healthy.
- Report macro-F1 and MCC, not accuracy alone.

In [ ]:
# CELL 1 - Install dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas numpy sentencepiece protobuf

import os, re, json, math, random, glob, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    subprocess.run(['nvidia-smi'], check=False)

PROJECT = '/content/DePlag_Fast_Local' if os.path.exists('/content') else os.getcwd()
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
for d in ['data', 'data/raw', 'models', 'reports', 'evaluation']:
    os.makedirs(d, exist_ok=True)
print('Working directory:', os.getcwd())
print('CUDA:', torch.cuda.is_available())

In [ ]:
# CELL 2 - Configuration

# Pretrained for paraphrase/duplicate-question classification. Much better starting point than roberta-base.
MODEL_NAME = 'cross-encoder/quora-roberta-base'

MAX_LENGTH = 128
EPOCHS = 1
LR = 1e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
WARMUP_RATIO = 0.06

# Caps keep the run fast. Increase only after a clean result.
MAX_TRAIN_EXAMPLES = 80000
MAX_VAL_EXAMPLES = 8000
MAX_TEST_EXAMPLES = 8000

# Local datasets. paraphrased_articles.csv is intentionally excluded.
USE_MSRP_LOCAL = True
USE_PAWS_LOCAL = True
USE_STS_LOCAL = True
USE_SNLI_LOCAL = True

# SNLI is not a plagiarism dataset. Use a capped auxiliary slice only.
SNLI_MAX_EXAMPLES = 40000

print({
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'lr': LR,
    'train_cap': MAX_TRAIN_EXAMPLES,
    'val_cap': MAX_VAL_EXAMPLES,
    'test_cap': MAX_TEST_EXAMPLES,
})

In [ ]:
# CELL 3 - Find local raw files

SEARCH_ROOTS = [
    Path('data/raw'),
    Path('/content/data/raw'),
    Path('/content'),
    Path('/kaggle/input'),
]

def find_file(name):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        direct = root / name
        if direct.exists():
            return str(direct)
        matches = list(root.rglob(name))
        if matches:
            return str(matches[0])
    return None

RAW_FILES = {
    'msrp': find_file('msr_paraphrase_train.txt'),
    'paws': find_file('paws_train.csv'),
    'sts': find_file('sts-train.jsonl'),
    'snli': find_file('snli_1.0_train.csv'),
    'excluded_recent': find_file('paraphrased_articles.csv'),
}
print(json.dumps(RAW_FILES, indent=2))
missing = [k for k, v in RAW_FILES.items() if k != 'excluded_recent' and v is None]
if missing:
    print('Missing local files:', missing)
    print('On Kaggle, add the dataset folder as an Input or upload these files under data/raw.')

In [ ]:
# CELL 4 - Load local datasets

def clean_text(x):
    if not isinstance(x, str):
        return ''
    return ' '.join(x.strip().split())

def norm_key(x):
    x = clean_text(x).lower()
    x = re.sub(r'[^a-z0-9\s]', '', x)
    return ' '.join(x.split())

def finalize(df, source):
    df = df[['label', 'text_a', 'text_b']].copy()
    df['text_a'] = df['text_a'].map(clean_text)
    df['text_b'] = df['text_b'].map(clean_text)
    df = df[(df.text_a != '') & (df.text_b != '')].copy()
    df['label'] = df['label'].astype(int)
    df = df[df['label'].isin([0, 1])].copy()
    df['source'] = source
    return df.reset_index(drop=True)

def load_msrp(path):
    raw = pd.read_csv(path, sep='\t', quoting=3, on_bad_lines='skip')
    cols = list(raw.columns)
    df = raw[[cols[0], cols[3], cols[4]]].copy()
    df.columns = ['label', 'text_a', 'text_b']
    return finalize(df, 'msrp_local')

def load_paws(path):
    raw = pd.read_csv(path, on_bad_lines='skip')
    if 'sentence1' not in raw.columns:
        raw = pd.read_csv(path, sep='\t', on_bad_lines='skip')
    df = raw[['label', 'sentence1', 'sentence2']].copy()
    df.columns = ['label', 'text_a', 'text_b']
    return finalize(df, 'paws_local')

def load_sts(path):
    rows = [json.loads(line) for line in open(path, encoding='utf-8', errors='ignore') if line.strip()]
    raw = pd.DataFrame(rows)
    score_col = 'score' if 'score' in raw.columns else 'similarity_score'
    s1_col = 'sentence1'
    s2_col = 'sentence2'
    df = pd.DataFrame({
        'text_a': raw[s1_col],
        'text_b': raw[s2_col],
        'label': pd.to_numeric(raw[score_col], errors='coerce').map(lambda v: 1 if v >= 3.5 else 0),
    })
    return finalize(df, 'sts_local')

def load_snli(path, max_examples):
    usecols = ['gold_label', 'sentence1', 'sentence2']
    chunks = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=100000, on_bad_lines='skip'):
        chunk = chunk[chunk.gold_label.isin(['entailment', 'neutral', 'contradiction'])].copy()
        chunk['label'] = (chunk.gold_label == 'entailment').astype(int)
        chunk = chunk.rename(columns={'sentence1': 'text_a', 'sentence2': 'text_b'})[['label', 'text_a', 'text_b']]
        chunks.append(chunk)
        if sum(len(c) for c in chunks) >= max_examples * 3:
            break
    raw = finalize(pd.concat(chunks, ignore_index=True), 'snli_local')
    pos = raw[raw.label == 1]
    neg = raw[raw.label == 0]
    per_class = min(len(pos), len(neg), max_examples // 2)
    raw = pd.concat([
        pos.sample(per_class, random_state=SEED),
        neg.sample(per_class, random_state=SEED),
    ], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    return raw

parts = []
if USE_MSRP_LOCAL and RAW_FILES['msrp']:
    parts.append(load_msrp(RAW_FILES['msrp']))
if USE_PAWS_LOCAL and RAW_FILES['paws']:
    parts.append(load_paws(RAW_FILES['paws']))
if USE_STS_LOCAL and RAW_FILES['sts']:
    parts.append(load_sts(RAW_FILES['sts']))
if USE_SNLI_LOCAL and RAW_FILES['snli']:
    parts.append(load_snli(RAW_FILES['snli'], SNLI_MAX_EXAMPLES))

assert parts, 'No local datasets were loaded.'
df = pd.concat(parts, ignore_index=True)
print('Loaded rows:', len(df))
print(df.source.value_counts())
print(df.label.value_counts())

In [ ]:
# CELL 5 - Dedupe, balance, split

a = df['text_a'].map(norm_key)
b = df['text_b'].map(norm_key)
df['pair_key'] = np.minimum(a, b) + ' ||| ' + np.maximum(a, b)
before = len(df)
df = df.drop_duplicates('pair_key').reset_index(drop=True)
print({'before_dedupe': before, 'after_dedupe': len(df)})

def balanced_sample(frame, max_examples, name):
    counts = frame.label.value_counts().to_dict()
    assert 0 in counts and 1 in counts, f'{name} missing a class: {counts}'
    per_class = min(counts[0], counts[1], max_examples // 2)
    out = pd.concat([
        frame[frame.label == 0].sample(per_class, random_state=SEED),
        frame[frame.label == 1].sample(per_class, random_state=SEED),
    ], ignore_index=True)
    return out.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Split before final balancing to avoid duplicated examples crossing splits.
from sklearn.model_selection import train_test_split
trainval_df, test_df = train_test_split(df, test_size=0.12, stratify=df.label, random_state=SEED)
train_df, val_df = train_test_split(trainval_df, test_size=0.12, stratify=trainval_df.label, random_state=SEED)

train_df = balanced_sample(train_df, MAX_TRAIN_EXAMPLES, 'train')
val_df = balanced_sample(val_df, MAX_VAL_EXAMPLES, 'validation')
test_df = balanced_sample(test_df, MAX_TEST_EXAMPLES, 'test')

train_keys = set(train_df.pair_key)
val_keys = set(val_df.pair_key)
test_keys = set(test_df.pair_key)
print('leak train-val:', len(train_keys & val_keys))
print('leak train/test or val/test:', len((train_keys | val_keys) & test_keys))

for name, split in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print('\n' + name, len(split))
    print(split.label.value_counts())
    print(split.source.value_counts())

train_df.drop(columns=['pair_key']).to_csv('data/train_pairs.csv', index=False)
val_df.drop(columns=['pair_key']).to_csv('data/val_pairs.csv', index=False)
test_df.drop(columns=['pair_key']).to_csv('data/test_pairs.csv', index=False)

In [ ]:
# CELL 6 - Tokenize

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def to_ds(frame):
    return Dataset.from_pandas(frame[['text_a', 'text_b', 'label', 'source']].reset_index(drop=True), preserve_index=False)

raw_ds = DatasetDict({
    'train': to_ds(train_df),
    'validation': to_ds(val_df),
    'test': to_ds(test_df),
})

def tok(batch):
    return tokenizer(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LENGTH)

encoded = raw_ds.map(tok, batched=True, remove_columns=['text_a', 'text_b', 'source'])
encoded = encoded.rename_column('label', 'labels')
print(encoded)

In [ ]:
# CELL 7 - Metrics

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef, precision_recall_curve, classification_report, confusion_matrix

def logits_to_proba(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = np.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(np.clip(logits, -50, 50))
    proba = exp_logits[:, 1] / np.clip(exp_logits.sum(axis=1), 1e-12, None)
    return np.nan_to_num(proba, nan=0.5, posinf=1.0, neginf=0.0)

def metric_dict(y_true, proba, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    proba = np.nan_to_num(np.asarray(proba), nan=0.5, posinf=1.0, neginf=0.0)
    pred = (proba >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
    macro = precision_recall_fscore_support(y_true, pred, average='macro', zero_division=0)[2]
    out = {
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'macro_f1': float(macro),
        'mcc': float(matthews_corrcoef(y_true, pred)),
        'threshold': float(threshold),
    }
    if len(set(y_true)) == 2:
        out['roc_auc'] = float(roc_auc_score(y_true, proba))
    return out

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metric_dict(labels, logits_to_proba(logits), threshold=0.5)

def best_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-9, None)
    idx = int(np.nanargmax(f1))
    return float(thresholds[idx]), float(f1[idx])

In [ ]:
# CELL 8 - Train fast high-accuracy cross-encoder

from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.use_cache = False

collator = DataCollatorWithPadding(tokenizer=tokenizer)
steps_per_epoch = max(1, math.ceil(len(encoded['train']) / TRAIN_BATCH_SIZE))
warmup_steps = int(steps_per_epoch * EPOCHS * WARMUP_RATIO)
major_cc = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
USE_BF16 = bool(torch.cuda.is_available() and major_cc >= 8)
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)
print({'model': MODEL_NAME, 'fp16': USE_FP16, 'bf16': USE_BF16, 'steps_per_epoch': steps_per_epoch, 'warmup_steps': warmup_steps})

args = TrainingArguments(
    output_dir='models/fast_cross_encoder',
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    max_grad_norm=1.0,
    optim='adamw_torch',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=100,
    logging_nan_inf_filter=True,
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded['train'],
    eval_dataset=encoded['validation'],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()
trainer.save_model('models/best_fast_cross_encoder')
tokenizer.save_pretrained('models/best_fast_cross_encoder')

In [ ]:
# CELL 9 - Final evaluation

def predict_split(split):
    pred = trainer.predict(encoded[split])
    labels = np.asarray(pred.label_ids).astype(int)
    proba = logits_to_proba(pred.predictions)
    print(split, {'n': len(labels), 'finite': bool(np.isfinite(proba).all()), 'min': float(proba.min()), 'max': float(proba.max())})
    return labels, proba

y_val, val_proba = predict_split('validation')
threshold, val_f1 = best_threshold(y_val, val_proba)
val_metrics = metric_dict(y_val, val_proba, threshold)

y_test, test_proba = predict_split('test')
test_metrics = metric_dict(y_test, test_proba, threshold)
test_pred = (test_proba >= threshold).astype(int)

print('Validation threshold:', threshold)
print('Validation:', json.dumps(val_metrics, indent=2))
print('Test:', json.dumps(test_metrics, indent=2))
print(classification_report(y_test, test_pred, target_names=['Not Plagiarism', 'Plagiarism']))

results = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'train_examples': len(train_df),
    'validation_examples': len(val_df),
    'test_examples': len(test_df),
    'local_files': RAW_FILES,
    'excluded_files': ['paraphrased_articles.csv'],
    'validation': val_metrics,
    'test': test_metrics,
    'confusion_matrix': confusion_matrix(y_test, test_pred).tolist(),
    'classification_report': classification_report(y_test, test_pred, target_names=['Not Plagiarism', 'Plagiarism'], output_dict=True),
}
with open('reports/results.json', 'w') as f:
    json.dump(results, f, indent=2)
with open('reports/classifier_report.txt', 'w') as f:
    f.write(json.dumps(results, indent=2))
pd.DataFrame({'label': y_test, 'proba': test_proba, 'pred': test_pred}).to_csv('reports/test_predictions.csv', index=False)
print('Saved reports/results.json, reports/classifier_report.txt, reports/test_predictions.csv')

## If This Still Runs Too Long

Do not switch to a weaker raw base model. First reduce `MAX_TRAIN_EXAMPLES` to `40000` and keep the pretrained cross-encoder. If metrics stay healthy, scale back up.